## [프로젝트] Seq2Seq으로 한국어 번역기 만들기

In [1]:
!pip install torch
!pip install mecab-python3

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
import re
import MeCab
import time

# ==========================================
# GPU/CPU 디바이스 설정
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"현재 사용 중인 디바이스: {device}")

현재 사용 중인 디바이스: cuda


In [3]:
# ==========================================
# Step 1 & 2: 데이터 파일 로드 및 정제
# ==========================================
print("데이터 로드 및 정제 시작...")

path_ko = 'work/korean-english-park.train/korean-english-park.train.ko'
path_en = 'work/korean-english-park.train/korean-english-park.train.en'

with open(path_ko, 'r', encoding='utf-8') as f:
    ko_raw = f.read().splitlines()
with open(path_en, 'r', encoding='utf-8') as f:
    en_raw = f.read().splitlines()

# 중복 제거 (set 활용, 병렬성 유지)
raw_corpus = list(set(zip(ko_raw, en_raw)))
print(f"중복 제거 후 데이터 수: {len(raw_corpus)}")

def preprocess_sentence(sentence, is_english=False):
    sentence = sentence.lower().strip()
    sentence = re.sub(r"([?.!,])", r" \1 ", sentence)
    sentence = re.sub(r'[" "]+', " ", sentence)
    sentence = re.sub(r"[^a-zA-Zㄱ-ㅎ가-힣?.!,]+", " ", sentence)
    sentence = sentence.strip()
    
    if is_english:
        sentence = '<start> ' + sentence + ' <end>'
    return sentence

mecab = MeCab.Tagger()
kor_corpus = []
eng_corpus = []

for ko, en in raw_corpus:
    ko_pre = preprocess_sentence(ko, is_english=False)
    en_pre = preprocess_sentence(en, is_english=True)
    
    parsed = mecab.parse(ko_pre)
    ko_tokens = [line.split('\t')[0] for line in parsed.split('\n') if line not in ['EOS', '']]
    en_tokens = en_pre.split()
    
    if len(ko_tokens) <= 40 and len(en_tokens) <= 40:
        kor_corpus.append(" ".join(ko_tokens))
        eng_corpus.append(en_pre)

print(f"길이 40 이하 필터링 후 데이터 수: {len(kor_corpus)}")

데이터 로드 및 정제 시작...
중복 제거 후 데이터 수: 78968
길이 40 이하 필터링 후 데이터 수: 62733


In [5]:
# ==========================================
# Step 3: 데이터 토큰화 및 텐서 변환 (Custom Vocab)
# ==========================================

class Vocabulary:
    def __init__(self, num_words=15000):
        # 0: 패딩(빈칸 채우기용), 1: OOV(사전에 없는 단어 처리용)
        self.word2idx = {"<pad>": 0, "<unk>": 1}
        self.idx2word = {0: "<pad>", 1: "<unk>"}
        self.num_words = num_words  # 단어장 최대 크기 설정
        self.idx = 2                # 실제 단어에 부여할 시작 번호

    def fit(self, sentences):
        # 1. 모든 문장을 순회하며 단어별 빈도수 계산
        word_counts = {}
        for sentence in sentences:
            for word in sentence.split():
                word_counts[word] = word_counts.get(word, 0) + 1
        
        # 2. 빈도수가 높은 순서대로 단어 정렬 (가장 자주 쓰이는 단어 위주로 사전 구성)
        sorted_words = sorted(word_counts.items(), key=lambda x: x[1], reverse=True)
        
        # 3. 빈도순 상위 단어들을 사전(self.word2idx)에 등록
        for word, _ in sorted_words:
            if len(self.word2idx) >= self.num_words: # 최대 단어 수 도달 시 중단
                break
            if word not in self.word2idx:
                self.word2idx[word] = self.idx
                self.idx2word[self.idx] = word
                self.idx += 1

    def texts_to_sequences(self, sentences):
        # 각 문장의 단어들을 사전의 인덱스(숫자)로 변환
        return [[self.word2idx.get(word, 1) for word in sentence.split()] for sentence in sentences]

def pad_sequences(sequences, padding='post'):
    # 배치 내 문장들의 길이를 가장 긴 문장에 맞추어 0(패딩)으로 채움
    max_len = max(len(seq) for seq in sequences)
    # PyTorch의 CrossEntropyLoss는 int64(LongTensor) 형태의 입력을 기대하므로 dtype 설정
    padded = np.zeros((len(sequences), max_len), dtype=np.int64) 
    
    for i, seq in enumerate(sequences):
        if padding == 'post': # 문장 뒤에 패딩 추가
            padded[i, :len(seq)] = seq
        else:                # 문장 앞에 패딩 추가
            padded[i, -len(seq):] = seq
    return padded

# ==========================================
# 토크나이저 훈련 및 텐서 변환 실행
# ==========================================
NUM_WORDS = 15000

# 한국어 데이터 사전 생성 및 인덱스 변환
ko_vocab = Vocabulary(NUM_WORDS)
ko_vocab.fit(kor_corpus)
ko_seqs = ko_vocab.texts_to_sequences(kor_corpus)
ko_tensor = pad_sequences(ko_seqs, padding='post') # 모델 입력용 텐서 생성

# 영어 데이터 사전 생성 및 인덱스 변환
en_vocab = Vocabulary(NUM_WORDS)
en_vocab.fit(eng_corpus)
en_seqs = en_vocab.texts_to_sequences(eng_corpus)
en_tensor = pad_sequences(en_seqs, padding='post') # 모델 학습용 타겟 텐서 생성

SRC_VOCAB_SIZE = len(ko_vocab.word2idx)
TGT_VOCAB_SIZE = len(en_vocab.word2idx)

print(f"한국어 단어 사전 크기: {SRC_VOCAB_SIZE}")
print(f"영어 단어 사전 크기: {TGT_VOCAB_SIZE}")

한국어 단어 사전 크기: 15000
영어 단어 사전 크기: 15000


In [9]:
# ==========================================
# Step 4: 모델 설계 (Bahdanau Attention 기반 Seq2seq)
# ==========================================

# 1. 모델 학습에 필요한 하이퍼파라미터 설정
BATCH_SIZE = 128      # 한 번에 묶어서 학습할 데이터(문장)의 개수
EMBEDDING_DIM = 256   # 단어를 컴퓨터가 이해할 수 있게 표현할 벡터의 차원 크기
HIDDEN_UNITS = 512    # GRU(RNN) 내부의 은닉 상태(Hidden State) 차원 크기 (모델의 기억 용량)

class Encoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim, enc_units):
        super(Encoder, self).__init__()
        self.enc_units = enc_units
        
        # 단어의 고유 인덱스를 밀집 벡터(Dense Vector)로 변환하는 임베딩 층
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        
        # 시계열 데이터를 처리하는 GRU 층 
        # (batch_first=True: 입력 데이터의 첫 번째 차원이 BATCH_SIZE임을 알려줌)
        self.gru = nn.GRU(embedding_dim, enc_units, batch_first=True)

    def forward(self, x, hidden):
        # 1. 입력된 단어 인덱스(x)를 임베딩 벡터로 변환
        x = self.embedding(x)
        
        # 2. 임베딩된 벡터와 이전 은닉 상태(hidden)를 GRU에 통과시킴
        # output: 문장의 모든 단어에 대한 결과물 (이후 Attention 계산 시 핵심 자료로 쓰임)
        # state: 문장을 끝까지 읽고 난 후의 최종 요약본 (디코더의 초기 상태로 넘어감)
        output, state = self.gru(x, hidden)
        return output, state

class BahdanauAttention(nn.Module):
    def __init__(self, units):
        super(BahdanauAttention, self).__init__()
        # 어텐션 스코어(어떤 단어에 집중할지 점수)를 계산하기 위한 3개의 선형 층(Linear Layer)
        self.W1 = nn.Linear(units, units) # 인코더의 출력을 변환하기 위한 가중치
        self.W2 = nn.Linear(units, units) # 디코더의 현재 상태를 변환하기 위한 가중치
        self.V = nn.Linear(units, 1)      # 변환된 값들을 최종 1차원 점수(Score)로 압축

    def forward(self, hidden, enc_output):
        # hidden shape 차원 변경: (1, batch_size, hidden_size) -> (batch_size, 1, hidden_size)
        # 인코더 출력(enc_output)과 행렬 덧셈을 하기 위해 차원의 위치를 맞춰주는 작업
        hidden_with_time_axis = hidden.permute(1, 0, 2)
        
        # 1. 스코어(Score) 계산: 디코더의 현재 상태와 인코더의 출력들을 조합하여 연관성 점수를 구함
        score = self.V(torch.tanh(self.W1(enc_output) + self.W2(hidden_with_time_axis)))
        
        # 2. 어텐션 가중치(Weights) 계산: 점수에 Softmax를 씌워 전체 합이 1이 되는 확률 값으로 만듦 
        # (예: "대통령" 번역 시 "president" 단어에 90% 집중)
        attention_weights = torch.softmax(score, dim=1)
        
        # 3. 컨텍스트 벡터(Context Vector) 생성: 인코더의 모든 출력 정보에 가중치를 곱해 중요한 부분만 추출
        context_vector = attention_weights * enc_output
        context_vector = torch.sum(context_vector, dim=1) # 단어 길이 축을 기준으로 하나로 합침
        
        return context_vector, attention_weights

class Decoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim, dec_units):
        super(Decoder, self).__init__()
        self.dec_units = dec_units
        
        # 디코더에 입력되는 타겟 단어를 벡터로 변환
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        
        # 디코더의 GRU 층
        # ★핵심: 입력 차원이 (embedding_dim + dec_units) 입니다. 
        # 방금 예측한 단어와 어텐션으로 가져온 컨텍스트 벡터를 '함께' 입력받기 때문입니다.
        self.gru = nn.GRU(embedding_dim + dec_units, dec_units, batch_first=True)
        
        # 디코더의 결과를 단어장의 실제 단어로 매핑하는 출력 층
        self.fc = nn.Linear(dec_units, vocab_size)
        
        # 위에서 정의한 Bahdanau Attention 모듈을 가져와 사용
        self.attention = BahdanauAttention(dec_units)

    def forward(self, x, hidden, enc_output):
        # 1. 어텐션을 통해 원본 문장에서 현재 집중해야 할 정보(컨텍스트 벡터)를 가져옴
        context_vector, attention_weights = self.attention(hidden, enc_output)
        
        # 2. 현재 입력된 타겟 단어를 임베딩 벡터로 변환
        x = self.embedding(x)
        
        # 3. 컨텍스트 벡터와 임베딩된 단어를 하나로 이어붙임(Concatenate)
        # unsqueeze(1)을 통해 차원을 (batch_size, 1, hidden_size)로 맞춘 후 붙입니다.
        x = torch.cat((context_vector.unsqueeze(1), x), dim=-1)
        
        # 4. 결합된 정보를 디코더 GRU에 통과시켜 다음 상태를 계산
        output, state = self.gru(x, hidden)
        
        # 5. 불필요한 차원 제거: (batch_size, 1, hidden_size) -> (batch_size, hidden_size)
        output = output.squeeze(1)
        
        # 6. 최종 선형 층을 통과하여 단어장에 있는 단어들 중 어떤 단어가 나올지 확률 분포 계산
        x = self.fc(output)
        
        return x, state, attention_weights

# ==========================================
# 2. 인코더와 디코더 객체 생성 및 디바이스(GPU/CPU)에 올리기
# ==========================================
# 위에서 정의한 클래스들을 바탕으로 실제 모델을 생성하고 `.to(device)`로 GPU 메모리에 할당합니다.
encoder = Encoder(SRC_VOCAB_SIZE, EMBEDDING_DIM, HIDDEN_UNITS).to(device)
decoder = Decoder(TGT_VOCAB_SIZE, EMBEDDING_DIM, HIDDEN_UNITS).to(device)

In [ ]:
# ==========================================
# Step 5: 훈련하기 및 테스트
# ==========================================

import torch

# 이상 탐지 모드를 켜면 nan이 발생하는 정확한 레이어와 원인을 에러 메시지로 띄워줍니다.
torch.autograd.set_detect_anomaly(False)

# 패딩(0)은 손실 계산에서 제외
criterion = nn.CrossEntropyLoss(ignore_index=0) 
optimizer = optim.Adam(list(encoder.parameters()) + list(decoder.parameters()), lr=0.0005)

# DataLoader 생성
dataset = TensorDataset(torch.tensor(ko_tensor), torch.tensor(en_tensor))
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)

def train_step(inp, targ):
    loss = 0
    optimizer.zero_grad()
    
    # 초기 hidden state
    enc_hidden = torch.zeros(1, BATCH_SIZE, HIDDEN_UNITS).to(device)
    
    enc_output, enc_hidden = encoder(inp, enc_hidden)
    dec_hidden = enc_hidden
    
    # 디코더의 첫 입력은 <start> 토큰
    dec_input = torch.tensor([[en_vocab.word2idx['<start>']] * BATCH_SIZE]).squeeze(0).unsqueeze(1).to(device)
    
    # Teacher Forcing
    for t in range(1, targ.size(1)):
        predictions, dec_hidden, _ = decoder(dec_input, dec_hidden, enc_output)
        # 현재 타임스텝의 타겟이 모두 패딩(0)이 아닐 때만 Loss 누적
        if (targ[:, t] != 0).sum() > 0:
            loss += criterion(predictions, targ[:, t])
        dec_input = targ[:, t].unsqueeze(1) # 다음 스텝의 입력은 실제 정답 타겟
        
    batch_loss = loss.item() / int(targ.size(1))
    loss.backward()
    torch.nn.utils.clip_grad_norm_(encoder.parameters(), max_norm=1.0)
    torch.nn.utils.clip_grad_norm_(decoder.parameters(), max_norm=1.0)
    optimizer.step()
    
    return batch_loss

def translate(sentence):
    encoder.eval()
    decoder.eval()
    
    sentence = preprocess_sentence(sentence, is_english=False)
    parsed = mecab.parse(sentence)
    tokens = [line.split('\t')[0] for line in parsed.split('\n') if line not in ['EOS', '']]
    
    inputs = [ko_vocab.word2idx.get(i, ko_vocab.word2idx['<unk>']) for i in tokens]
    # 모델 추론 시 배치 사이즈는 1
    inputs = torch.tensor(inputs).unsqueeze(0).to(device)
    
    result = ''
    hidden = torch.zeros(1, 1, HIDDEN_UNITS).to(device)
    
    with torch.no_grad():
        enc_out, enc_hidden = encoder(inputs, hidden)
        dec_hidden = enc_hidden
        dec_input = torch.tensor([[en_vocab.word2idx['<start>']]]).to(device)
        
        max_len = en_tensor.shape[1]
        
        for t in range(max_len):
            predictions, dec_hidden, _ = decoder(dec_input, dec_hidden, enc_out)
            predicted_id = predictions.argmax(1).item()
            
            if en_vocab.idx2word[predicted_id] == '<end>':
                result += ' <end>'
                return result.strip()
            
            result += en_vocab.idx2word[predicted_id] + ' '
            dec_input = torch.tensor([[predicted_id]]).to(device)
            
    return result.strip()

EPOCHS = 50 
examples = [
    "오바마는 대통령이다.",
    "시민들은 도시 속에 산다.",
    "커피는 필요 없다.",
    "일곱 명의 사망자가 발생했다."
]

print("\n본격적인 학습을 시작합니다!")
for epoch in range(EPOCHS):
    start = time.time()
    total_loss = 0
    
    encoder.train()
    decoder.train()
    
    for batch_idx, (inp, targ) in enumerate(dataloader):
        inp, targ = inp.to(device), targ.to(device)
        batch_loss = train_step(inp, targ)
        total_loss += batch_loss

    print(f'Epoch {epoch + 1} Loss {total_loss / len(dataloader):.4f}')
    print(f'Time taken for 1 epoch {time.time() - start:.2f} sec\n')
    
    for i, ex in enumerate(examples):
        print(f"K{i+1}) {ex}")
        print(f"E{i+1}) {translate(ex)}")
    print("-" * 50)


본격적인 학습을 시작합니다!
Epoch 1 Loss 5.1397
Time taken for 1 epoch 163.06 sec

K1) 오바마는 대통령이다.
E1) the president s government , said he was not to the country .  <end>
K2) 시민들은 도시 속에 산다.
E2) the <unk> <unk> <unk> <unk> , <unk> said .  <end>
K3) 커피는 필요 없다.
E3) the <unk> , said the <unk> said .  <end>
K4) 일곱 명의 사망자가 발생했다.
E4) the u . s . military said .  <end>
--------------------------------------------------
Epoch 2 Loss 4.4097
Time taken for 1 epoch 161.21 sec

K1) 오바마는 대통령이다.
E1) president bush s president bush said he would not be a very important .  <end>
K2) 시민들은 도시 속에 산다.
E2) they were still in the <unk> , <unk> <unk> <unk> .  <end>
K3) 커피는 필요 없다.
E3) the <unk> is not to be the first time .  <end>
K4) 일곱 명의 사망자가 발생했다.
E4) the people were killed , police said .  <end>
--------------------------------------------------
Epoch 3 Loss 4.0104
Time taken for 1 epoch 160.96 sec

K1) 오바마는 대통령이다.
E1) obama is the first time , he said .  <end>
K2) 시민들은 도시 속에 산다.
E2) they were <unk> by the <unk> , 

### 회고  

##### - Epoch를 거듭할수록 Loss는 5.13에서 0.85까지 감소하였으나, 번역 결과물을 볼 때 아직 어색한 부분이 많이 보인다.  
##### 학습 횟수를 50회 이상 늘리게 되면 학습 효과가 더 좋아지겠지만 1Epoch당 3분씩 소요되기에 이 부분은 진행 해 보지 못하였다.  
##### - 다른 대안으로 **Teacher Forcing 비율 점진적 감소** 나 **양방향 인코더(Bidirectional GRU)** 를 적용하는 대안을 Ai가 제시하였으나,  
##### 시간 관계 상 여기까지는 진행하지 못한 게 아쉬움으로 남는다.  

